In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
df.info()
df.shape

In [ ]:
df['Churn'].value_counts()

In [ ]:
df.head(10)

In [ ]:
for i in df.columns:
    print("col=",i , df[i].unique())

In [ ]:
yes_no_cols = list()
for col in df.columns:
    if set(df[col].dropna().unique()).issubset({'Yes', 'No'}):
        yes_no_cols.append(col)
        

In [ ]:
for i in yes_no_cols:
    df[i] = df[i].map({'Yes':1, 'No':0})
    
df['gender'] = df['gender'].map({'Male': 1, "Female": 0})

In [ ]:
df.sample(10)

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df['TotalCharges'].fillna(0, inplace=True) 

In [ ]:
df.drop(columns=['customerID'], inplace=True)

In [ ]:


for col in df.columns:
    if df[col].nunique() > 2 and df[col].dtype == 'object': 
        print(col)

In [ ]:
df = pd.get_dummies(df, columns=[
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaymentMethod'], drop_first=True, dtype=int)

In [ ]:
df.sample(10)

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
# import matplotlib.pyplot as plt
# import pandas as pd
# import seaborn as sns

# # Create pairplot with hue
# sns.pairplot(df, hue="Churn")
# plt.show()

In [ ]:
r = df.corr(numeric_only=True)['Churn'].abs().sort_values()
r

In [ ]:
from ctf import CorrelationThresholdFilter
cr = CorrelationThresholdFilter(0.09)
cr.fit(df.drop(columns=['Churn']), df['Churn'])


In [ ]:
filltered_features = list()

for i in r.index: 
    if r[i] > 0.09 and i != "Churn":
        filltered_features.append(i)

In [ ]:
filltered_features

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [ ]:
from IPython.core import getipython
X = df[filltered_features]
y = df['Churn']

# 2. Train-Test Split (stratified to maintain churn ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Initialize and train Decision Tree
# max_depth prevents overfitting; class_weight='balanced' addresses churn imbalance
dt_model = DecisionTreeClassifier(
    max_depth=4, random_state=42
)
dt_model.fit(X_train, y_train)

# 4. Predict and Evaluate
# y_prob = (dt_model.predict_proba(X_test) >= 0.65 ).astype(int)[:, 1]
# y_prob = dt_model.predict_proba(X_test)[:,1]

y_pred = (dt_model.predict_proba(X_test)[:, 1] >= 0.45).astype(int)

# print(y_prob)
# print('AUC-ROC Score:', round(roc_auc_score(y_test, y_prob), 4))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

# 5. Visualize the Tree
# plt.figure(figsize=(20, 10))
# plot_tree(
#     dt_model,
#     feature_names=filltered_features,
#     class_names=['No Churn', 'Churn'],
#     filled=True,
#     rounded=True,
#     fontsize=9,
# )
# plt.tight_layout()
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[filltered_features]
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression( max_iter=1000, random_state=42
)
lr_model.fit(X_train_scaled, y_train)

# 5. Predict & Evaluate
y_pred = lr_model.predict(X_test_scaled)
y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]



print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
for i, col in enumerate(['tenure', 'MonthlyCharges', 'TotalCharges'], 1):
    plt.subplot(1, 3, i)
    plt.boxplot(df[col].dropna())
    plt.title(col)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.pipeline import Pipeline
from cleaningcls import clean_cls

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier())
])